[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLManuals/Deep-Learning/blob/main/chapter05_loss_functions/notebook.ipynb)

# Chapter 5: Loss Functions

Run the cell below once, first, to install any missing packages (only needed on Google Colab; skip it if you're running locally and already have them).

In [ ]:
%pip install -q torch numpy scikit-learn

On the first night of 1801, the Italian astronomer Giuseppe Piazzi was mapping stars from his observatory in Palermo when he noticed a faint point of light that was not on any chart. Over the following nights it moved, slowly, against the background stars. Piazzi suspected he had found something new, perhaps even a missing planet that astronomers had been hunting for years, and he tracked it carefully for about forty days. Then it slipped into the glare of the sun and disappeared from view.

The object, which came to be called Ceres, would come out from behind the sun months later, but nobody knew where to look. Piazzi's few weeks of measurements covered only a tiny arc of its path across the sky, and every one of them contained small errors, because no telescope and no human eye is perfect. The best astronomers in Europe tried to work out its orbit, and their predictions disagreed with each other by large amounts.

A young German mathematician named Carl Friedrich Gauss, then in his early twenties, took a different approach. He accepted that every measurement was a little wrong, and instead of trusting any single one he looked for the orbit that disagreed with all of them as little as possible overall. To measure that disagreement, he squared each error and added the squares together, and then chose the orbit that made the total as small as he could. On the seventh of December 1801, the astronomer Franz Xaver von Zach pointed his telescope at the spot Gauss had calculated, and there was Ceres, very close to where Gauss had said it would be. The method, published by Adrien Marie Legendre in 1805 and by Gauss in 1809, is called **least squares**, and you have used it every time you trained a linear regression.

Gauss's real breakthrough was not a formula for orbits. It was the idea of turning "how wrong is this answer" into a single number, and then searching for the answer that makes that number as small as possible. A neural network learns in exactly the same way, and this chapter is about that number, which is called the **loss**. Before you read on, try to guess the answers to these four questions.

1. **If a network overestimates on one day and underestimates on another, why can you not just add up its errors?**
2. **Why might squaring an error be better than simply ignoring its sign?**
3. **How should you score a forecaster who said there was a 99 percent chance of sunshine on a day it poured?**
4. **Is a loss of 0.69 good or bad?**

## Measuring How Wrong a Network Is

A network that is learning needs feedback, and the feedback has to be a single number. If it got back a long list of mistakes, it would have no way to tell whether a change to its weights had made things better overall or worse. The loss is that single number. It looks at every guess the network made, compares each one with the right answer, and squeezes all of those comparisons into one value, where a smaller value means a better network and zero means a perfect one.

Think of the loss as the score in a game of darts where lower is better. Each dart lands some distance from the bullseye, and the loss turns all of those distances into one number, with each loss function using its own scoring rule.

Imagine a small tea shop that wants to predict how many cups of tea it will sell each morning, so that it brews the right amount. The first cell holds what really happened over five mornings, along with one set of guesses.

In [ ]:
import numpy as np

actual = np.array([42, 35, 50, 28, 46])    # cups of tea sold
guess = np.array([47, 30, 50, 28, 46])

The **error** for each morning is the guess minus the actual number, so a positive error means the shop brewed too much and a negative error means it ran short. The obvious first idea is to add the errors up.

In [ ]:
errors = guess - actual
print(errors)
print(errors.sum())

The total is zero, as if every guess were perfect, yet the shop brewed five cups too many on the first morning and ran five short on the second. The two mistakes cancelled out, and a loss that hides real mistakes is useless. **A loss must never let mistakes in opposite directions cancel each other out, so every error has to count as a cost, whichever way it points.** The two most common ways of doing that are to ignore the sign of each error or to square it.

## Mean Squared Error for Numbers

Ignoring the sign gives the **mean absolute error**, or MAE, which is simply the average size of the misses. For the guesses above, the MAE is 2 cups, because the misses were 5, 5, 0, 0 and 0. Squaring each error before averaging gives the **mean squared error**, or MSE, which is the loss Gauss used and the most common loss for networks that predict numbers.

Suppose the tea shop compares two different ways of guessing. Model A is never exactly right but never far wrong, while model B is perfect on four mornings and badly wrong on the fifth, when a coach party arrived and the shop ran out of tea. Table 5.1 shows their guesses side by side, and the next cell stores both sets of guesses.

**Table 5.1: Two ways of guessing the same five mornings. Model A makes small misses every day, while model B is perfect on four days and badly wrong on one.**

| Morning | Actual | Model A | Error A | Model B | Error B |
|---|---|---|---|---|---|
| 1 | 42 | 40 | $-2$ | 42 | 0 |
| 2 | 35 | 38 | 3 | 35 | 0 |
| 3 | 50 | 47 | $-3$ | 50 | 0 |
| 4 | 28 | 30 | 2 | 28 | 0 |
| 5 | 46 | 44 | $-2$ | 35 | $-11$ |

In [ ]:
guess_a = np.array([40, 38, 47, 30, 44])
guess_b = np.array([42, 35, 50, 28, 35])

The loop below scores each model in turn, printing its MAE first and its MSE second.

In [ ]:
for g in (guess_a, guess_b):
    err = g - actual
    print(np.abs(err).mean(), (err ** 2).mean())

By the MAE, model B looks slightly better, with an average miss of 2.2 cups against 2.4. By the MSE, model A is four times better, with 6.0 against 24.2. Squaring makes the single miss of 11 cups count as 121, while each of model A's small misses counts only 4 or 9. **The mean squared error punishes one large mistake far more heavily than several small ones, which is usually what you want when a big miss is costly.** For a tea shop that loses customers when it runs out, model A really is the better choice, and the MSE is the loss that says so.

One drawback of the MSE is that its units are squared, so an MSE of 24.2 means 24.2 "squared cups", which is hard to picture. Taking the square root gives the **root mean squared error**, or RMSE, which is back in cups. For model A it is about 2.45 cups and for model B about 4.92. Networks are trained on the MSE because it is simpler to work with, and people report the RMSE because it is easier to read.

The MSE has one more property that matters enormously for the next chapter. Picture a café that guesses its cold drink sales by multiplying the day's temperature by a single weight $w$. The next cell holds six days of temperatures and sales, and computes the MSE for five different values of $w$.

In [ ]:
temp = np.array([20, 24, 28, 30, 33, 35])
drinks = np.array([41, 47, 58, 59, 67, 69])

def mse(w):
    return ((w * temp - drinks) ** 2).mean()

for w in [1.5, 1.8, 2.0, 2.1, 2.5]:
    print(w, round(mse(w), 1))

The loss falls as $w$ approaches 2 and climbs again beyond it. Figure 5.1 draws the MSE for every weight in that range, and the shape is a smooth bowl with a single lowest point. Finding the best weight means finding the bottom of the bowl, and Chapter 6 does exactly that by rolling downhill.

> 🖼️ **Figure 5.1:** The mean squared error of the cold drinks guess for every weight between 1.4 and 2.6. The loss forms a smooth bowl, and the best weight sits at its lowest point.

## Cross Entropy for Classes

Not every network predicts a number. A network that decides whether an email is spam, or whether a photograph shows a cat, a dog or a rabbit, gives a probability for each possible answer, and it needs a loss that scores probabilities. Think of a weather forecaster. One who said there was a 60 percent chance of sunshine on a day that turned out rainy was wrong, but reasonably modest about it. One who said 99 percent and was wrong deserves far more blame, because they were wrong and completely sure of themselves.

The loss that captures this is called **cross entropy**. Its rule is simple, which is to take the probability the network gave to the right answer and score it as minus the logarithm of that probability. The next cell scores six different probabilities for the correct answer, from very confident and right to very confident and wrong, and for comparison it also prints the squared error that the MSE would give.

In [ ]:
p = np.array([0.99, 0.9, 0.6, 0.5, 0.1, 0.01])
print((-np.log(p)).round(3))
print(((1 - p) ** 2).round(3))

When the network gives the right answer a probability of 0.99, the cross entropy is a tiny 0.01. As the probability falls the loss grows, slowly at first and then very fast, so that a probability of 0.01 for the right answer, which means the network was 99 percent sure of something false, costs 4.605. The squared error, in the second line, can never be larger than 1, so it barely tells a confidently wrong network apart from a merely unsure one. Figure 5.2 shows both penalties.

> 🖼️ **Figure 5.2:** Two ways of scoring a probability. Cross entropy grows without limit as a network becomes confident in a wrong answer, while the squared error never goes above 1.

When there are more than two classes, the network first needs a way to turn its raw output numbers, called **scores**, into probabilities that are all positive and add up to 1. The function that does this is called **softmax**. It raises the number $e$, about 2.718, to the power of each score, which makes every value positive and makes larger scores stand out, and then divides each result by their total.

In [ ]:
def softmax(scores):
    e = np.exp(scores)
    return e / e.sum()

scores = np.array([2.0, 1.0, 0.1])    # cat, dog, rabbit
probs = softmax(scores)
print(probs.round(3))

Figure 5.3 follows these three scores through softmax. If the photograph really shows a cat, the cross entropy uses only the cat's probability of 0.659, and if it shows a rabbit, it uses only the rabbit's probability of 0.099.

In [ ]:
print(round(-np.log(probs[0]), 3), round(-np.log(probs[2]), 3))

> 🖼️ **Figure 5.3:** Softmax turns three raw scores into three probabilities. The largest score keeps the largest share, and the results always add up to exactly 1.

A network that gave the right answer only 0.099 pays a loss of 2.317, more than five times the loss for a well placed guess. **Cross entropy scores the probability a network gave to the right answer, and it punishes confident mistakes so heavily that a network quickly learns not to be sure of something false.** For yes or no questions the same idea uses the sigmoid from Chapter 4 in place of softmax, and is then often called **binary cross entropy**.

## Reading a Loss Value

A loss value on its own means very little. A cross entropy of 0.5 might be excellent for one problem and terrible for another, so the skill is knowing what to compare it with. The most useful comparison is with a network that has learned nothing at all.

For cross entropy, a network that has learned nothing gives every class the same probability. With two classes that probability is one half, and with ten classes it is one tenth. The next cell prints the cross entropy of those know nothing guesses.

In [ ]:
print(round(-np.log(1 / 2), 3), round(-np.log(1 / 10), 3))

These two numbers are worth remembering. If your yes or no network has a loss stuck near 0.693, or your ten class network sits near 2.303, it is doing no better than guessing. Look back at Chapter 4, where the sigmoid network on the plants finished with a loss of 0.686, almost exactly 0.693, which told you, without any other evidence, that it had learned almost nothing.

For the mean squared error the know nothing network always predicts the average of the targets, whatever the input. Its loss is the natural yardstick for any real model, as the next cell shows for the café's cold drinks.

In [ ]:
baseline = ((drinks - drinks.mean()) ** 2).mean()
print(round(baseline, 1), round(mse(2.0), 1))

Always guessing the average gives an MSE of 100.8, while multiplying the temperature by 2 gives 1.5, so the simple model has removed almost all of the error that guessing would leave. Table 5.2 gathers the landmark values to compare against.

**Table 5.2: Landmark loss values. A real network should sit well below the first column, and no real network ever reaches the second.**

| Loss and task | A network that learned nothing | A perfect network |
|---|---|---|
| Cross entropy, 2 classes | 0.693 | 0 |
| Cross entropy, 10 classes | 2.303 | 0 |
| Cross entropy, $k$ classes | the logarithm of $k$ | 0 |
| Mean squared error | the variance of the targets | 0 |

Two more habits will save you a great deal of confusion. First, only compare losses that were measured the same way, on the same data, with the same loss function, because an MSE in squared cups and a cross entropy cannot be compared at all. Second, keep separate eyes on the training loss and the validation loss, exactly as Books 4 and 5 taught you. A training loss that keeps falling while the validation loss starts rising is the classic sign of overfitting, which Chapter 15 tackles directly. **Read a loss by comparing it with the loss of a network that knows nothing, and trust only the loss measured on data the network never trained on.**

## Practice and Review

Reading about deep learning is only half of learning it. Pulling an idea back out of your memory makes it stick much better than reading it again, so this section gives you two kinds of practice, one for your hands and one for your memory.

### Try It Yourself

1. **Find the bottom of the bowl.** Compute `mse(w)` for every weight from 1.90 to 2.10 in steps of 0.01 using `np.arange`, and find which weight gives the smallest loss.
2. **Make a confident mistake.** Change the scores to `[5.0, 1.0, 0.1]`, compute softmax again, and find the cross entropy if the photograph really shows a rabbit. Compare it with the 2.317 from this chapter.
3. **Check a baseline.** For a network that sorts photographs into 100 classes, work out the loss of a network that has learned nothing, and decide whether a validation loss of 4.5 is good news.

### Check Your Understanding

Try to answer each question from memory before you look at the answers that follow.

1. **Why can a loss not simply add up the raw errors?**
2. **Why did the MSE prefer model A while the MAE slightly preferred model B?**
3. **What shape does the MSE make when you change a single weight, and why does that matter?**
4. **Why is cross entropy a better loss for classes than squared error?**
5. **A yes or no network has a loss of 0.69 after training. What does that tell you?**

### Answers

1. Because errors in opposite directions cancel out, so a set of guesses with real mistakes could still add up to zero.
2. Because squaring makes model B's single miss of 11 cups cost 121, far more than all of model A's small misses together.
3. A smooth bowl with one lowest point, so finding the best weight means finding the bottom of the bowl.
4. Because it grows without limit as a network becomes confident in a wrong answer, while squared error can never exceed 1.
5. That it is doing no better than guessing, since 0.693 is the loss of giving both answers a probability of one half.

## What Comes Next

You can now measure how wrong a network is, choose the right loss for numbers or for classes, and read a loss value against an honest baseline.

- **Chapter 6, Gradient Descent From Scratch,** rolls down the bowl from this chapter to find the best weight automatically.
- **Chapter 7, Backpropagation From the Ground Up,** shows how a network works out which way to roll for every weight at once.

Chapter 6 begins in 1847, with a French mathematician who found a hard answer by stepping, again and again, in the steepest downhill direction.